# Session 11 — AI-Powered Applications

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tech4alltraining/aiml/blob/main/mlai-genai-internship/student/notebooks/session-11-ai-apps.ipynb)

**ML/AI & GenAI Internship** · Application concepts · Retrieval & RAG · GenAI with Streamlit · ML + GenAI integration

---

## How to use this notebook

**Everything in this course has been building to this session.** Session 5 gave you a model that decides. Session 10 gave you a model that writes. Here you join them.

| # | Topic |
|---|---|
| 1 | AI-powered Application Concepts |
| 2 | Retrieval and RAG |
| 3 | Building GenAI applications with Streamlit |
| 4 | Integrating Machine Learning with Generative AI |

> Streamlit apps are `.py` files run from a terminal, so those cells are **shown, not executed**. Everything else runs here.

Exercises, MCQs and tasks: [Session 11 guide](../sessions/session-11-ai-apps.md).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, json
import numpy as np
import pandas as pd

HAS_KEY = bool(os.environ.get("GEMINI_API_KEY"))
print("API key set:", HAS_KEY)
print("The API cells print the prompt they WOULD send when no key is set.")

API key set: False
The API cells print the prompt they WOULD send when no key is set.


---
# 1. AI-powered Application Concepts

**An AI-powered application is not "a chatbot with a wrapper".** It is ordinary software in which one component happens to be a model.

🧠 **A database-backed web app.** The database is essential, but nobody calls it "a database application" — it is a shop, or a booking system, that *uses* a database. **Treat your model the same way.**

| Pattern | What it does | Example |
|---|---|---|
| **Assistant** | Answers questions conversationally | A support chatbot |
| **Transformer** | Text in one form → another | Summariser, translator |
| **Extractor** | Unstructured text → structured data | CV → JSON of skills |
| **Generator** | A brief → new content | Product descriptions |
| **Augmenter** | Adds a layer to an existing system | **Your Session 5 model, explained** |

> **The last one is the most valuable and the least demonstrated.**

In [2]:
# The architecture that actually ships.
steps = [
    ("1. INPUT",    "validate before you spend a token"),
    ("2. RETRIEVE", "fetch relevant context (Topic 2)"),
    ("3. PROMPT",   "build it from a template"),
    ("4. MODEL",    "call the LLM / the ML model"),
    ("5. PARSE",    "JSON, with a fallback that works"),
    ("6. VALIDATE", "is the output actually usable?"),
    ("7. DISPLAY",  "with an honest confidence signal"),
    ("8. LOG",      "what was asked, what came back"),
]
for s, d in steps:
    print(f"  {s:<13}{d}")

print("\nSteps 1, 5, 6 and 8 are what separate a DEMO from an APPLICATION.")
print("The model call is the easy part.")

  1. INPUT     validate before you spend a token
  2. RETRIEVE  fetch relevant context (Topic 2)
  3. PROMPT    build it from a template
  4. MODEL     call the LLM / the ML model
  5. PARSE     JSON, with a fallback that works
  6. VALIDATE  is the output actually usable?
  7. DISPLAY   with an honest confidence signal
  8. LOG       what was asked, what came back

Steps 1, 5, 6 and 8 are what separate a DEMO from an APPLICATION.
The model call is the easy part.


In [3]:
# Two lines of validation prevent the two most common production incidents.
def validate(q):
    q = (q or "").strip()
    if not q:         return None, "Please type a question."
    if len(q) < 5:    return None, "Please ask a fuller question."
    if len(q) > 2000: return None, "That is too long - please shorten it."
    return q, None

for t in ["", "hi", "x" * 3000, "What is the refund policy?"]:
    label = repr(t[:24] + ("..." if len(t) > 24 else ""))
    print(f"{label:<32}{validate(t)[1] or 'OK -> call the model'}")

print("\nEmpty prompts burning quota, and a pasted document costing more")
print("than expected. Both stopped before any API call happens.")

''                              Please type a question.
'hi'                            Please ask a fuller question.
'xxxxxxxxxxxxxxxxxxxxxxxx...'   That is too long - please shorten it.
'What is the refund polic...'   OK -> call the model

Empty prompts burning quota, and a pasted document costing more
than expected. Both stopped before any API call happens.


In [4]:
# Parsing that survives contact with reality.
def parse_json(text, fallback=None):
    for candidate in (text,
                      text.strip().removeprefix("```json")
                          .removeprefix("```").removesuffix("```")):
        try:
            return json.loads(candidate)
        except (json.JSONDecodeError, AttributeError):
            continue
    return fallback

tests = ['{"sentiment": "positive"}',
         '```json\n{"sentiment": "positive"}\n```',
         'Sorry, I cannot help with that.']
for t in tests:
    print(f"{t[:36]!r:<42} -> {parse_json(t, fallback={'error': True})}")

print("\nEven with response_mime_type set, a defensive parse costs nothing")
print("and saves an outage. Write it once, reuse it everywhere.")
print()
print("An LLM NEVER says 'I do not know' unless you ask it to. Its job is")
print("to produce PLAUSIBLE text, not TRUE text. Design for that.")

'{"sentiment": "positive"}'                -> {'sentiment': 'positive'}
'```json\n{"sentiment": "positive"}\n``'   -> {'sentiment': 'positive'}
'Sorry, I cannot help with that.'          -> {'error': True}

Even with response_mime_type set, a defensive parse costs nothing
and saves an outage. Write it once, reuse it everywhere.

An LLM NEVER says 'I do not know' unless you ask it to. Its job is
to produce PLAUSIBLE text, not TRUE text. Design for that.


---
# 2. Retrieval and RAG

🧠 **A brilliant graduate with no access to your files.** They can reason superbly about anything you show them — and they have never seen your company handbook. **Asking them your refund policy from memory is unfair to them and dangerous for you. Hand them the handbook first.**

```text
1. RETRIEVE   find the documents relevant to the question
2. AUGMENT    paste them into the prompt as context
3. GENERATE   ask the model to answer USING ONLY that context
```

In [5]:
# The obvious first attempt: keyword matching. Watch it break.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DOCS = [
    "Refunds are available within 30 days of purchase with a valid receipt.",
    "Our office hours are Monday to Friday, 9am to 6pm IST.",
    "Shipping within India takes 3-5 working days. International orders take 10-14 days.",
    "To reset your password, click 'Forgot password' on the login page.",
    "Warranty covers manufacturing defects for 12 months from the date of purchase.",
    "We accept UPI, credit cards, debit cards and net banking.",
    "Damaged items must be reported within 48 hours of delivery with photographs.",
]
vec = TfidfVectorizer(stop_words="english")
M = vec.fit_transform(DOCS)

def retrieve(q, k=1):
    s = cosine_similarity(vec.transform([q]), M).ravel()
    return [(float(s[i]), DOCS[i]) for i in np.argsort(-s)[:k]]

print("QUESTIONS THAT SHARE WORDS WITH THE DOCUMENTS")
print("-" * 62)
for q in ["how do I reset my password?", "how long does shipping take?",
          "what is the refund policy?"]:
    sc, doc = retrieve(q)[0]
    mark = "OK " if sc > 0 else "FAIL"
    print(f"{mark} {sc:.3f}  {q}")
    print(f"          -> {doc[:60]}")

QUESTIONS THAT SHARE WORDS WITH THE DOCUMENTS
--------------------------------------------------------------
OK  0.707  how do I reset my password?
          -> To reset your password, click 'Forgot password' on the login
OK  0.305  how long does shipping take?
          -> Shipping within India takes 3-5 working days. International 
FAIL 0.000  what is the refund policy?
          -> Refunds are available within 30 days of purchase with a vali


In [6]:
print("QUESTIONS THAT MEAN THE SAME THING IN DIFFERENT WORDS")
print("-" * 62)
for q in ["how long do I have to return something?",
          "when are you open?",
          "my package arrived broken"]:
    sc, doc = retrieve(q)[0]
    mark = "OK " if sc > 0 else "FAIL"
    print(f"{mark} {sc:.3f}  {q}")

print()
print("EVERY ONE SCORES ZERO - and the answers are all in the documents.")
print("  'return'  is not 'refund'")
print("  'open'    is not 'office hours'")
print("  'broken'  is not 'damaged'")
print()
print("Worse: 'what is the REFUND policy?' also scored 0.000, against a")
print("document that literally says 'REFUNDS are available'. TF-IDF treats")
print("refund and refunds as two unrelated strings. Even a PLURAL breaks it.")
print()
print("THIS is why RAG uses EMBEDDINGS rather than keywords. An embedding")
print("maps text to a vector where MEANING determines position, so")
print("'broken' lands near 'damaged' even sharing no letters.")
print("Keyword search matches STRINGS; embeddings match MEANING.")

QUESTIONS THAT MEAN THE SAME THING IN DIFFERENT WORDS
--------------------------------------------------------------
FAIL 0.000  how long do I have to return something?
FAIL 0.000  when are you open?
FAIL 0.000  my package arrived broken

EVERY ONE SCORES ZERO - and the answers are all in the documents.
  'return'  is not 'refund'
  'open'    is not 'office hours'
  'broken'  is not 'damaged'

Worse: 'what is the REFUND policy?' also scored 0.000, against a
document that literally says 'REFUNDS are available'. TF-IDF treats
refund and refunds as two unrelated strings. Even a PLURAL breaks it.

THIS is why RAG uses EMBEDDINGS rather than keywords. An embedding
maps text to a vector where MEANING determines position, so
'broken' lands near 'damaged' even sharing no letters.
Keyword search matches STRINGS; embeddings match MEANING.


In [7]:
# The grounded prompt. Numbering the context is what makes citation possible.
RAG_PROMPT = """Answer the question using ONLY the context below.
If the context does not contain the answer, say exactly:
"I don't have that information."
After your answer, cite which context items you used.

Context:
{context}

Question: {question}
"""

q = "how long does shipping take?"
ctx = "\n".join(f"[{i+1}] {d}" for i, (_, d) in enumerate(retrieve(q, k=3)))
print(RAG_PROMPT.format(context=ctx, question=q))

print("-" * 62)
print("The 'say exactly' line is doing real work. Without it the model")
print("invents an answer rather than admitting the gap.")

Answer the question using ONLY the context below.
If the context does not contain the answer, say exactly:
"I don't have that information."
After your answer, cite which context items you used.

Context:
[1] Shipping within India takes 3-5 working days. International orders take 10-14 days.
[2] Refunds are available within 30 days of purchase with a valid receipt.
[3] Our office hours are Monday to Friday, 9am to 6pm IST.

Question: how long does shipping take?

--------------------------------------------------------------
The 'say exactly' line is doing real work. Without it the model
invents an answer rather than admitting the gap.


### The full pipeline

```text
BUILD (once)
  documents -> split into chunks -> embed each chunk -> store the vectors

ANSWER (per question)
  question -> embed -> find the nearest chunks -> paste into the prompt
           -> "answer using ONLY this context" -> cite the sources
```

| Decision | Typical choice | Why it matters |
|---|---|---|
| Chunk size | 200–500 words | Too big wastes context; too small loses meaning |
| Chunks retrieved | 3–5 | Too many buries the answer (Session 10's middle-of-context problem) |
| "Only use context" | **Always** | The difference between grounded and invented |
| Cite sources | **Always** | Lets a human check you |

```python
# Embeddings, with an API key:
result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=["my package arrived broken",
              "Damaged items must be reported within 48 hours."],
)
# The two vectors sit CLOSE TOGETHER despite sharing no words.
```

---
# 3. Building GenAI applications with Streamlit

You built a prediction app in Session 5. **A chat app needs one extra idea: memory.**

🧠 **Recall Streamlit's core behaviour:** the whole script reruns on every interaction. **For a chat app that means your conversation would vanish on every message** — unless you keep it in `st.session_state`.

```python
# chat_app.py
import streamlit as st
from google import genai

st.title("Study Assistant")

@st.cache_resource
def get_client():
    return genai.Client(api_key=st.secrets["GEMINI_API_KEY"])
client = get_client()

# 1. MEMORY - survives the rerun
if "messages" not in st.session_state:
    st.session_state.messages = []

# 2. REPLAY the whole conversation on every rerun
for m in st.session_state.messages:
    with st.chat_message(m["role"]):
        st.markdown(m["content"])

# 3. TAKE new input
if prompt := st.chat_input("Ask me anything about the course"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            reply = client.models.generate_content(
                model="gemini-3.5-flash", contents=prompt).text
        st.markdown(reply)
    st.session_state.messages.append({"role": "assistant", "content": reply})
```

```bash
streamlit run chat_app.py
```

**A complete chat application in about 30 lines.** The three numbered steps are the whole pattern: **remember, replay, append.**

### Streaming — which changes the whole feel

```python
with st.chat_message("assistant"):
    box = st.empty()
    full = ""
    for chunk in client.models.generate_content_stream(
            model="gemini-3.5-flash", contents=prompt):
        full += chunk.text or ""
        box.markdown(full + "▌")     # a blinking cursor
    box.markdown(full)
```

**A ten-second wait feels broken. Ten seconds of text appearing feels fast.** Same duration, completely different experience.

### The secrets file

```toml
# .streamlit/secrets.toml   <- add this to .gitignore
GEMINI_API_KEY = "your-key-here"
```

> ⚠️ **Never commit a key.** Automated scanners find keys pushed to GitHub within minutes.

### Real memory

```python
@st.cache_resource
def get_chat():
    return get_client().chats.create(model="gemini-3.5-flash")

reply = get_chat().send_message(prompt).text     # history travels with it
```

> ⚠️ **`@st.cache_resource` on the chat means every visitor shares one conversation.** Fine for a demo; wrong for anything public.

---
# 4. Integrating Machine Learning with Generative AI

**The most valuable pattern in the course.**

Your Session 5 model produces `0.87`. **A person cannot act on `0.87`.**

🧠 **A blood test and a doctor.** The lab returns numbers — precise, objective, meaningless to you. The doctor reads them and says *"your iron is low, here is what to do."* **The lab does not guess and the doctor does not measure.**

```text
┌─────────────┐     ┌──────────────┐     ┌─────────────┐
│  ML MODEL   │ --> │   THE NUMBER │ --> │     LLM     │ --> a person
│  decides    │     │   0.87       │     │  explains   │
└─────────────┘     └──────────────┘     └─────────────┘
   objective          the evidence          in English
   auditable                                 actionable
```

> ⚠️ **The LLM must never make the decision.** It explains a decision already made by a model you have measured, cross-validated and bootstrapped.

In [8]:
# STEP 1 - the ML model decides. Objective, measured, reproducible.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

BASE = "https://raw.githubusercontent.com/tech4alltraining/aiml/refs/heads/main/datasets/"
L = pd.read_csv(BASE + "loan_data_10k.csv").dropna().reset_index(drop=True)
for c in L.select_dtypes(include="object").columns:
    L[c] = LabelEncoder().fit_transform(L[c])
X, y = L.drop(columns=["loan_status"]), L["loan_status"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xtr, ytr)

row = Xte.iloc[[3]]
proba = model.predict_proba(row)[0][1]
decision = "approved" if proba > .5 else "declined"
print(f"ML DECISION: {decision}  (confidence {proba:.0%})")
print(f"model test accuracy: {model.score(Xte, yte):.4f}")

ML DECISION: approved  (confidence 100%)
model test accuracy: 0.8910


In [9]:
# STEP 2 - gather the evidence, FROM THE MODEL ITSELF.
imp = pd.Series(model.feature_importances_, index=X.columns)
top3 = imp.nlargest(3)
print("the model weighted these most heavily:")
print(top3.round(4).to_string())
print("\nThis is Topic 2's lesson applied to numbers: GROUND the")
print("explanation in what the model actually used, not in a guess.")

the model weighted these most heavily:
previous_loan_defaults_on_file    0.3461
loan_int_rate                     0.1543
loan_percent_income               0.1252

This is Topic 2's lesson applied to numbers: GROUND the
explanation in what the model actually used, not in a guess.


In [10]:
# STEP 3 - the LLM explains, using ONLY what it was given.
vals = {k: row.iloc[0][k] for k in top3.index}
prompt = f"""You are a loan officer writing to an applicant.

DECISION: {decision} (confidence {proba:.0%})
The model weighted these factors most heavily: {top3.round(3).to_dict()}
This applicant's values for those factors: {vals}

Write 3-4 sentences explaining this decision kindly and clearly.
Do NOT change the decision. Do NOT invent factors not listed above.
End with one specific, actionable suggestion.
"""
print(prompt)
print("=" * 66)
print("The two 'Do NOT' lines are the WHOLE SAFETY DESIGN.")
print()
print("Without them the model will soften a rejection into an approval,")
print("or invent a reason that sounds plausible and is not in your data.")
print("Language models are trained to be agreeable - that is a real and")
print("documented failure mode here.")

if HAS_KEY:
    from google import genai
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    print("\n--- explanation ---")
    print(client.models.generate_content(
        model="gemini-3.5-flash", contents=prompt).text)
else:
    print("\n[no API key set - add one to see the generated explanation]")

You are a loan officer writing to an applicant.

DECISION: approved (confidence 100%)
The model weighted these factors most heavily: {'previous_loan_defaults_on_file': 0.346, 'loan_int_rate': 0.154, 'loan_percent_income': 0.125}
This applicant's values for those factors: {'previous_loan_defaults_on_file': np.float64(0.0), 'loan_int_rate': np.float64(11.83), 'loan_percent_income': np.float64(0.41)}

Write 3-4 sentences explaining this decision kindly and clearly.
Do NOT change the decision. Do NOT invent factors not listed above.
End with one specific, actionable suggestion.

The two 'Do NOT' lines are the WHOLE SAFETY DESIGN.

Without them the model will soften a rejection into an approval,
or invent a reason that sounds plausible and is not in your data.
Language models are trained to be agreeable - that is a real and
documented failure mode here.

[no API key set - add one to see the generated explanation]


In [11]:
# Where each piece of the value comes from - and why it MUST.
rows = [
    ("the decision",    "Random Forest",        "auditable, consistent, measured"),
    ("the confidence",  "predict_proba",        "a real number, not a vibe"),
    ("the factors",     "feature_importances_", "grounded in the actual model"),
    ("the explanation", "the LLM",              "the only part it is uniquely good at"),
]
print(f"{'piece':<18}{'comes from':<24}{'why it must'}")
print("-" * 82)
for a_, b_, c_ in rows:
    print(f"{a_:<18}{b_:<24}{c_}")

print(f"\n{'':<20}{'ML model':<14}{'LLM'}")
print("-" * 48)
for label, ml, llm in [("Decides", "YES", "NEVER"), ("Auditable", "YES", "no"),
                       ("Consistent", "YES", "no"), ("Explains in English", "no", "YES"),
                       ("Handles the unexpected", "no", "YES")]:
    print(f"{label:<20}{ml:<14}{llm}")

print("\nAsk the LLM to DECIDE and you throw away every guarantee")
print("Sessions 5-8 gave you: the cross-validated score, the confidence")
print("interval, the auditability, and even consistency - the same")
print("applicant could get different answers on different days.")

piece             comes from              why it must
----------------------------------------------------------------------------------
the decision      Random Forest           auditable, consistent, measured
the confidence    predict_proba           a real number, not a vibe
the factors       feature_importances_    grounded in the actual model
the explanation   the LLM                 the only part it is uniquely good at

                    ML model      LLM
------------------------------------------------
Decides             YES           NEVER
Auditable           YES           no
Consistent          YES           no
Explains in English no            YES
Handles the unexpectedno            YES

Ask the LLM to DECIDE and you throw away every guarantee
Sessions 5-8 gave you: the cross-validated score, the confidence
interval, the auditability, and even consistency - the same
applicant could get different answers on different days.


### The whole application

```python
# app.py
import streamlit as st, joblib, pandas as pd
from google import genai

st.title("Loan Decision Assistant")

@st.cache_resource
def load():
    return joblib.load("loan_pipeline.joblib"), genai.Client(
        api_key=st.secrets["GEMINI_API_KEY"])
pipeline, client = load()

income = st.number_input("Annual income", 0, 1_000_000, 50_000, step=1_000)
amount = st.number_input("Loan amount", 0, 500_000, 10_000, step=1_000)
score  = st.slider("Credit score", 300, 850, 650)

if st.button("Assess"):
    row = pd.DataFrame([{"person_income": income, "loan_amnt": amount,
                         "credit_score": score}])
    proba = pipeline.predict_proba(row)[0][1]          # ML DECIDES
    decision = "approved" if proba > .5 else "declined"

    st.metric("Decision", decision.title(), f"{proba:.0%} confidence")

    with st.spinner("Writing explanation..."):         # LLM EXPLAINS
        explanation = client.models.generate_content(
            model="gemini-3.5-flash",
            contents=f'''Explain this loan decision in 3-4 kind sentences.
DECISION: {decision} (confidence {proba:.0%})
Income {income}, loan {amount}, credit score {score}.
Do NOT change the decision. Do NOT invent factors.
End with one actionable suggestion.''').text
    st.write(explanation)

    st.caption("Educational demo. Decisions are made by a statistical model "
               "and should be reviewed by a person.")
```

**This is a complete capstone-grade deliverable.**

---
# ✅ Before you move on

- [ ] Naming the five application patterns and picking between them
- [ ] Knowing the LLM is a component, not the application
- [ ] Validating input **before** spending a token
- [ ] Explaining hallucination and why grounding fixes it
- [ ] Having seen keyword search fail on paraphrases, and why embeddings fix it
- [ ] Describing the RAG pipeline end to end
- [ ] Building a Streamlit chat app with memory, streaming and a clear button
- [ ] Never committing an API key
- [ ] **The ML model decides and the LLM explains — never the reverse**
- [ ] Designing an application that fails safely

**Exercises, MCQs and tasks:** [Session 11 guide](../sessions/session-11-ai-apps.md)

| | |
|---|---|
| **Previous** | [Session 10 — GenAI & LLMs](../sessions/session-10-genai-llms.md) |
| **Tutorials** | [AI-powered apps](../tutorials/concepts/ai-powered-apps.md) · [ML + GenAI](../tutorials/apps/ml_gen_ai.md) |
| **Stuck?** | [Troubleshooting](../troubleshooting.md) |